# Portfolio Allocation — FIN 3700 Final Exam Figures

**Universe:** PLTR, NVDA, INTC, GLD, EEM &nbsp;(SPY as market benchmark)

**Two-period structure:**

| Window  | Range | Purpose |
|---------|-------|---------|
| Training | Nov 2020 – Apr 2025 (~4.5 yrs) | Estimate E[r], σ, Σ, β, α; solve for GMV and tangency weights |
| Holding  | May 2025 – Apr 2026 (~12 mo)   | Apply training-window weights out-of-sample |

Every statistic, table, and figure is produced for **both** windows side-by-side so the
exam can probe in-sample vs. out-of-sample reasoning on the same constructs.

**Output files** (in `exam_figures/`):

| Kind | Training | Holding |
|------|----------|---------|
| Asset summary stats | `tab1a_asset_stats_training.html` | `tab1b_asset_stats_holding.html` |
| Covariance matrix   | `tab3a_cov_training.html`         | `tab3b_cov_holding.html`         |
| Portfolio stats     | `tab2a_portfolio_stats_training.html` | `tab2b_portfolio_stats_holding.html` |
| Efficient frontier  | `fig1a_frontier_training.png`     | `fig1b_frontier_holding.png`     |
| Cumulative wealth   | `fig2_cumwealth_training.png`     | `fig3_cumwealth_holding.png`     |
| Frontier composition| `fig4a_composition_training.png`  | `fig4b_composition_holding.png`  |
| SCL + SML           | `fig5a_scl_sml_training.png`      | `fig5b_scl_sml_holding.png`      |

All long-only — short-selling is intentionally excluded for this exam.

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cvxpy as cp
from dotenv import load_dotenv
from tiingo import TiingoClient
from fredapi import Fred

load_dotenv()

# --- Universe ---
TICKERS = ['SPY', 'PLTR', 'NVDA', 'INTC', 'GLD', 'EEM']
ASSETS  = ['PLTR', 'NVDA', 'INTC', 'GLD', 'EEM']  # portfolio universe (excludes benchmark)

# --- Date windows ---
FETCH_START = '2020-10-01'   # PLTR IPO'd 2020-09-30; gives a clean Nov-2020 start
FETCH_END   = '2026-04-30'

TRAIN_START = '2020-11-01'
TRAIN_END   = '2025-04-30'
HOLD_START  = '2025-05-01'
HOLD_END    = '2026-04-30'

TRADING_DAYS = 252

# --- Directories ---
OUTDIR    = 'exam_figures'
CACHE_DIR = 'data_cache'
os.makedirs(OUTDIR,    exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

PRICES_CACHE = os.path.join(CACHE_DIR, f'prices_{FETCH_START}_{FETCH_END}.parquet')
FRED_CACHE   = os.path.join(CACHE_DIR, f'fred_rf_{FETCH_START}_{FETCH_END}.parquet')

# --- Save helper: consistent PNG export across every figure ---
def savefig(fig, name):
    path = os.path.join(OUTDIR, name)
    fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f'  saved -> {path}')

# --- Title helper: bold title + gray italic subtitle (no em-dash) ---
def set_titled(ax, title, subtitle=None, pad=20):
    """
    Set a bold left-aligned title with an optional gray-italic subtitle below it.
    Replaces the 'Title — Subtitle' em-dash pattern.
    """
    ax.set_title(title, fontsize=12, fontweight='bold', loc='left', pad=pad)
    if subtitle:
        ax.text(0.0, 1.02, subtitle, transform=ax.transAxes,
                fontsize=10, style='italic', color='#666666')


## 2. Data Retrieval

Parquet cache lives in `data_cache/`. On the first run the API gets hit; every run after that loads from disk. To force a refresh, delete the relevant `.parquet` file (or the whole `data_cache/` directory).

The cache filename includes the fetch date range so changing `FETCH_END` automatically triggers a re-pull.

In [ ]:
# --- Risk-free rate from FRED (3-month Treasury) ---
if os.path.exists(FRED_CACHE):
    print(f"Loading FRED cache: {FRED_CACHE}")
    fred_rf = pd.read_parquet(FRED_CACHE)
else:
    print("Fetching 3-Month Treasury from FRED...")
    fred = Fred(api_key=os.getenv("FRED_API_KEY"))
    fred_series = fred.get_series('DGS3MO',
                                  observation_start=FETCH_START,
                                  observation_end=FETCH_END)
    fred_rf = pd.DataFrame(fred_series, columns=['DAILY_RF'])
    fred_rf['DAILY_RF'] = fred_rf['DAILY_RF'] / 100 / TRADING_DAYS   # annualized % -> daily decimal
    fred_rf.index.name = 'date'
    fred_rf.to_parquet(FRED_CACHE)
    print(f"  cached -> {FRED_CACHE}")


In [ ]:
# --- Equity prices from Tiingo ---
if os.path.exists(PRICES_CACHE):
    print(f"Loading prices cache: {PRICES_CACHE}")
    prices = pd.read_parquet(PRICES_CACHE)
else:
    print("Fetching equity prices from Tiingo...")
    client = TiingoClient({'session': True, 'api_key': os.getenv("TIINGO_API_KEY")})
    prices = pd.DataFrame()
    for ticker in TICKERS:
        data = client.get_dataframe(ticker, startDate=FETCH_START, endDate=FETCH_END)
        prices[ticker] = data['adjClose']
        print(f"  pulled {ticker}")
    prices.index = prices.index.tz_localize(None)
    prices.index.name = 'date'
    prices.to_parquet(PRICES_CACHE)
    print(f"  cached -> {PRICES_CACHE}")


In [ ]:
# --- Daily simple returns, merged with risk-free, split into training & holding ---
returns_all = prices.pct_change().dropna()
returns_all = returns_all.join(fred_rf, how='left').ffill().dropna()

returns_train = returns_all.loc[TRAIN_START:TRAIN_END].copy()
returns_hold  = returns_all.loc[HOLD_START:HOLD_END].copy()

prices_train = prices.loc[TRAIN_START:TRAIN_END]
prices_hold  = prices.loc[HOLD_START:HOLD_END]

print(f"Training window: {returns_train.index[0].date()} -> {returns_train.index[-1].date()}  ({len(returns_train)} days)")
print(f"Holding window:  {returns_hold.index[0].date()} -> {returns_hold.index[-1].date()}  ({len(returns_hold)} days)")


## 3. Student Reference Portfolios

The four portfolios students built during the semester project, reconstructed on the exam universe. Each is a fixed weight vector — independent of the optimizer.

- **Equal-Weight** — wᵢ = 1/N
- **Price-Weight** — wᵢ ∝ Pᵢ at the start of the training window (Dow-style)
- **Value-Weight** — wᵢ ∝ market cap (snapshot near training start; edit `MARKET_CAPS_BN` to use a different vintage)
- **Bottom-Up** — instructor reference fundamental allocation; edit `w_bottomup_dict` for the answer key

In [ ]:
# Equal-Weight
w_equal = np.ones(len(ASSETS)) / len(ASSETS)

# Price-Weight (proportional to start-of-training-window price)
p0 = prices_train[ASSETS].iloc[0].values
w_price = p0 / p0.sum()

# Value-Weight: hardcoded approximate $B market caps near Nov 2020
MARKET_CAPS_BN = {
    'PLTR':  45.0,
    'NVDA': 330.0,
    'INTC': 205.0,
    'GLD':   75.0,
    'EEM':   30.0,
}
mcap_arr = np.array([MARKET_CAPS_BN[a] for a in ASSETS], dtype=float)
w_value = mcap_arr / mcap_arr.sum()

# Bottom-Up — instructor reference
w_bottomup_dict = {
    'PLTR': 0.20,
    'NVDA': 0.35,
    'INTC': 0.05,
    'GLD':  0.20,
    'EEM':  0.20,
}
w_bottomup = np.array([w_bottomup_dict[a] for a in ASSETS])
assert abs(w_bottomup.sum() - 1.0) < 1e-9, "Bottom-Up weights must sum to 1"

# Display
student_weights = pd.DataFrame(
    np.column_stack([w_bottomup, w_equal, w_price, w_value]),
    index=ASSETS,
    columns=['Bottom-Up', 'Equal-Weight', 'Price-Weight', 'Value-Weight']
)
student_weights.style.format('{:.2%}')\
    .background_gradient(cmap='Blues', axis=None)\
    .set_caption('Student Reference Portfolios')


## 4. Per-Window Helper Functions

Everything downstream is parameterized by a returns window so we can produce training and holding versions side-by-side without duplicating code.

In [ ]:
def window_label(ret_window):
    """Date-range string for subtitles."""
    return f"{ret_window.index[0].date()} to {ret_window.index[-1].date()}"

def asset_summary(ret_window):
    """Per-asset stats over the given window: E[r], σ, β, α (annual), α t-stat, Sharpe."""
    summary = pd.DataFrame(index=ASSETS)
    mu_daily  = ret_window[ASSETS].mean()
    sigma_daily = ret_window[ASSETS].std()
    rf_daily  = ret_window['DAILY_RF'].mean()
    rf_ann    = rf_daily * TRADING_DAYS

    summary['E[r]'] = mu_daily * TRADING_DAYS
    summary['σ']    = sigma_daily * np.sqrt(TRADING_DAYS)

    market_excess = (ret_window['SPY'] - ret_window['DAILY_RF']).rename('SPY')
    X = sm.add_constant(market_excess)

    alphas_ann, alpha_t, betas = [], [], []
    for a in ASSETS:
        y = ret_window[a] - ret_window['DAILY_RF']
        res = sm.OLS(y, X).fit()
        alphas_ann.append(res.params['const'] * TRADING_DAYS)
        alpha_t.append(res.tvalues['const'])
        betas.append(res.params['SPY'])

    summary['β']        = betas
    summary['α']        = alphas_ann   # annualized
    summary['α t-stat'] = alpha_t
    summary['Sharpe']   = (summary['E[r]'] - rf_ann) / summary['σ']

    return summary, rf_ann, X

def style_summary(summary, subtitle):
    return (summary.style
        .format({
            'E[r]':     '{:.2%}',
            'σ':        '{:.2%}',
            'β':        '{:.3f}',
            'α':        '{:.2%}',
            'α t-stat': '{:.2f}',
            'Sharpe':   '{:.3f}',
        })
        .background_gradient(cmap='Blues',  subset=['E[r]', 'σ'])
        .background_gradient(cmap='RdYlGn', subset=['α t-stat'], vmin=-3, vmax=3)
        .background_gradient(cmap='Greens', subset=['Sharpe'])
        .set_caption(f'<div style="font-weight:bold;font-size:1.05em;">Asset Summary Statistics</div>'
                     f'<div style="font-style:italic;font-size:0.85em;color:#666;">{subtitle}</div>'))

def style_cov(cov, subtitle):
    return (cov.style
        .format('{:.6f}')
        .background_gradient(cmap='Blues', axis=None)
        .set_caption(f'<div style="font-weight:bold;font-size:1.05em;">Daily Covariance Matrix</div>'
                     f'<div style="font-style:italic;font-size:0.85em;color:#666;">{subtitle}</div>'))


## 5. Asset Summary Statistics

Per-asset annualized E[r], σ, β, α, α t-stat, and Sharpe. Computed independently on each window. The lookup table for most exam questions.

In [ ]:
# --- Training window ---
summary_train, rf_train_ann, X_train = asset_summary(returns_train)
mu_train_daily = returns_train[ASSETS].mean()
rf_train_daily = returns_train['DAILY_RF'].mean()
betas_train     = summary_train['β'].tolist()
alphas_train_ann = summary_train['α'].tolist()

styled_summary_train = style_summary(summary_train, f"Training Window: {window_label(returns_train)}")
with open(os.path.join(OUTDIR, 'tab1a_asset_stats_training.html'), 'w') as f:
    f.write(styled_summary_train.to_html())
print(f"  saved -> {os.path.join(OUTDIR, 'tab1a_asset_stats_training.html')}")
styled_summary_train


In [ ]:
# --- Holding window ---
summary_hold, rf_hold_ann, X_hold = asset_summary(returns_hold)
mu_hold_daily = returns_hold[ASSETS].mean()
rf_hold_daily = returns_hold['DAILY_RF'].mean()
betas_hold     = summary_hold['β'].tolist()
alphas_hold_ann = summary_hold['α'].tolist()

styled_summary_hold = style_summary(summary_hold, f"Holding Window: {window_label(returns_hold)}")
with open(os.path.join(OUTDIR, 'tab1b_asset_stats_holding.html'), 'w') as f:
    f.write(styled_summary_hold.to_html())
print(f"  saved -> {os.path.join(OUTDIR, 'tab1b_asset_stats_holding.html')}")
styled_summary_hold


## 6. Covariance Matrices

Daily covariance over each window. The training matrix feeds the optimizer; comparing to the holding matrix is how students see that diversification structure isn't static.

In [ ]:
# --- Training ---
cov_train = returns_train[ASSETS].cov()
styled_cov_train = style_cov(cov_train, f"Training Window: {window_label(returns_train)}")
with open(os.path.join(OUTDIR, 'tab3a_cov_training.html'), 'w') as f:
    f.write(styled_cov_train.to_html())
print(f"  saved -> {os.path.join(OUTDIR, 'tab3a_cov_training.html')}")
styled_cov_train


In [ ]:
# --- Holding ---
cov_hold = returns_hold[ASSETS].cov()
styled_cov_hold = style_cov(cov_hold, f"Holding Window: {window_label(returns_hold)}")
with open(os.path.join(OUTDIR, 'tab3b_cov_holding.html'), 'w') as f:
    f.write(styled_cov_hold.to_html())
print(f"  saved -> {os.path.join(OUTDIR, 'tab3b_cov_holding.html')}")
styled_cov_hold


## 7. Portfolio Optimization (long-only)

Two long-only solvers via convex QP:

- **GMV** — minimize σ²(w) subject to Σwᵢ = 1, w ≥ 0
- **Tangency** — maximize Sharpe (non-convex directly); solved via the standard reformulation as min yᵀΣy s.t. (μ − r_f·1)ᵀy = 1, y ≥ 0, then renormalize y so weights sum to 1.

In [ ]:
def gmv_cvx(cov, lb=0.0, ub=1.0):
    n = cov.shape[0]
    w = cp.Variable(n)
    prob = cp.Problem(cp.Minimize(cp.quad_form(w, cp.psd_wrap(cov.values))),
                      [cp.sum(w) == 1, w >= lb, w <= ub])
    prob.solve()
    return np.array(w.value)

def tangency_cvx(mu, cov, rf, lb=0.0, ub=1.0):
    n = cov.shape[0]
    excess = mu.values - rf
    y = cp.Variable(n)
    prob = cp.Problem(cp.Minimize(cp.quad_form(y, cp.psd_wrap(cov.values))),
                      [excess @ y == 1, y >= lb])
    prob.solve()
    y_val = np.array(y.value)
    return y_val / y_val.sum()

def frontier_cvx(mu, cov, target_returns, lb=0.0, ub=1.0):
    n = cov.shape[0]
    mu_arr = mu.values
    vols, rets, ws = [], [], []
    for tr in target_returns:
        w = cp.Variable(n)
        prob = cp.Problem(cp.Minimize(cp.quad_form(w, cp.psd_wrap(cov.values))),
                          [cp.sum(w) == 1, mu_arr @ w == tr, w >= lb, w <= ub])
        try:
            prob.solve()
            if w.value is not None and prob.status in ('optimal', 'optimal_inaccurate'):
                w_val = np.array(w.value)
                vols.append(np.sqrt(w_val @ cov.values @ w_val))
                rets.append(tr)
                ws.append(w_val)
        except cp.SolverError:
            continue
    return np.array(vols), np.array(rets), np.array(ws)


In [ ]:
# --- Solve GMV & tangency on the training window (these are the 'committed' weights) ---
w_gmv = gmv_cvx(cov_train, lb=0.0, ub=1.0)
w_tan = tangency_cvx(mu_train_daily, cov_train, rf_train_daily, lb=0.0, ub=1.0)

# --- Also solve them on the holding window (what optimizer would have wanted in hindsight) ---
w_gmv_hold = gmv_cvx(cov_hold, lb=0.0, ub=1.0)
w_tan_hold = tangency_cvx(mu_hold_daily, cov_hold, rf_hold_daily, lb=0.0, ub=1.0)

for lbl, w in [('GMV (train)', w_gmv), ('Tangency (train)', w_tan),
               ('GMV (hold)', w_gmv_hold), ('Tangency (hold)', w_tan_hold)]:
    assert abs(w.sum() - 1) < 1e-6, f"{lbl} weights don't sum to 1"

opt_weights = pd.DataFrame(
    np.column_stack([w_gmv, w_tan, w_gmv_hold, w_tan_hold]),
    index=ASSETS,
    columns=pd.MultiIndex.from_tuples([
        ('Training', 'GMV'),      ('Training', 'Tangency'),
        ('Holding',  'GMV'),      ('Holding',  'Tangency'),
    ])
)
opt_weights.style.format('{:.2%}')\
    .background_gradient(cmap='Blues', axis=None)\
    .set_caption('Optimizer Weights — Training vs. Holding (hindsight)')


## 8. Figure 1 — Efficient Frontier

Long-only frontier with GMV, tangency, CAL, and the four student portfolios marked.

Two versions are produced:

- **Training-window frontier** — the (σ, E[r]) plane the optimizer "saw" when computing GMV and tangency weights. The training-derived `w_gmv` and `w_tan` sit exactly on this frontier.
- **Holding-window frontier** — the realized (σ, E[r]) plane during the holding period. The frontier is computed on holding-window data (where the optimizer *would have* wanted to be in hindsight). Training-derived `w_gmv` and `w_tan` are marked as hollow points showing where they actually landed; holding-window-optimal points are marked as filled hindsight markers.

Marker shapes are distinct so the figure prints cleanly in grayscale.

In [ ]:
import matplotlib as mpl

# ---- Theming: white background, dual-density grid ----
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.edgecolor':    '#333333',
    'axes.linewidth':    1.0,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.size':         10,
    'axes.grid':         False,   # we'll set up grids manually for major/minor
})

# ---- Magma-based palette for portfolio markers ----
# Six samples from magma in [0.15, 0.87] avoid both ends (too dark / too pale).
# Assignment order: GMV -> Tangency -> 4 student portfolios.
_magma = mpl.colormaps['magma']
_magma_samples = [_magma(p) for p in (0.15, 0.30, 0.45, 0.60, 0.75, 0.87)]

PORTFOLIO_COLORS = {
    'GMV':          _magma_samples[0],   # deep indigo
    'Tangency':     _magma_samples[1],   # purple
    'Bottom-Up':    _magma_samples[2],   # magenta
    'Equal-Weight': _magma_samples[3],   # rose
    'Price-Weight': _magma_samples[4],   # coral
    'Value-Weight': _magma_samples[5],   # peach
}

FRONTIER_COLOR  = '#1F2A6E'   # deep navy — cool, contrasts with warm magma points
DOMINATED_COLOR = '#7C8BB8'   # muted navy
CAL_COLOR       = '#333333'   # near-black

# ---- Per-item label offsets, hand-tuned to avoid overlap ----
# (dx, dy) in display points (~pixels) relative to the marker. Edit any entry
# to nudge a single label without touching the rest.
PORTFOLIO_LABEL_OFFSETS = {
    'GMV':          (-38, -16),   # below-left, clear of EEM and the y-axis
    'Tangency':     (10, 12),     # above-right
    'Bottom-Up':    (12,  8),     # above-right (clear of Tangency)
    'Equal-Weight': (14, -6),     # right, slightly below the marker
    'Price-Weight': (12, -14),    # below-right (separates from GLD)
    'Value-Weight': (10, 10),     # above-right (isolated)
}
ASSET_LABEL_OFFSETS = {
    'PLTR': (8,  6),
    'NVDA': (8,  6),
    'INTC': (10, 8),
    'GLD':  (-26, 10),            # upper-left to separate from Price-Weight
    'EEM':  (8,  8),
}

# ---- Grid helper: dual-density major + minor ----
def style_grid(ax):
    """Apply major (solid) and minor (dashed, fainter) gridlines."""
    ax.minorticks_on()
    ax.grid(which='major', color='#BBBBBB', linewidth=0.7, linestyle='-',  alpha=0.7, zorder=0)
    ax.grid(which='minor', color='#DDDDDD', linewidth=0.5, linestyle='--', alpha=0.6, zorder=0)

def annualize(vols, rets):
    return vols * np.sqrt(TRADING_DAYS), rets * TRADING_DAYS

def port_stats_window(w, ret_window):
    """(annual return, annual σ, annual Sharpe) for weights w applied to ret_window."""
    daily = ret_window[ASSETS] @ np.asarray(w)
    r = daily.mean() * TRADING_DAYS
    s = daily.std()  * np.sqrt(TRADING_DAYS)
    rf = ret_window['DAILY_RF'].mean() * TRADING_DAYS
    return r, s, (r - rf) / s

# ---- Student-portfolio plotting metadata ----
# Marker shape per portfolio; color comes from PORTFOLIO_COLORS so the magma
# palette stays the single source of truth.
STUDENT_PORTFOLIOS = [
    ('Bottom-Up',    w_bottomup, 'P'),
    ('Equal-Weight', w_equal,    'D'),
    ('Price-Weight', w_price,    '^'),
    ('Value-Weight', w_value,    'v'),
]

def plot_frontier(ret_window, mu_daily, cov, rf_ann, w_gmv_win, w_tan_win,
                  title_subtitle, savename,
                  extra_committed=None):
    """
    Plot a long-only efficient frontier for the given window.

    extra_committed: optional list of (label, w, marker) — drawn as HOLLOW markers
    in the matching PORTFOLIO_COLORS color to show where training-derived points
    land on the holding-window frontier. Pass None on the training-window plot.
    The base portfolio name is parsed from the label ('GMV (training-derived)'
    -> 'GMV') to pick the color.
    """
    eps = 1e-6
    tgt = np.linspace(mu_daily.min() + eps, mu_daily.max() - eps, 80)
    fv, fr, _ = frontier_cvx(mu_daily, cov, tgt, lb=0.0, ub=1.0)
    fv_ann, fr_ann = annualize(fv, fr)

    gmv_r, gmv_s, _          = port_stats_window(w_gmv_win, ret_window)
    tan_r, tan_s, tan_sharpe = port_stats_window(w_tan_win, ret_window)

    eff = fr_ann >= gmv_r

    fig, ax = plt.subplots(figsize=(12, 7.5))

    # Grid first so points/lines render on top
    style_grid(ax)

    # ---- Frontier curves ----
    ax.plot(fv_ann[eff],  fr_ann[eff],  '-',  color=FRONTIER_COLOR,  lw=2.4,
            label='Efficient frontier', zorder=3)
    ax.plot(fv_ann[~eff], fr_ann[~eff], '--', color=DOMINATED_COLOR, lw=1.2, alpha=0.7,
            label='Dominated portfolios', zorder=3)

    # ---- CAL through this window's tangency ----
    x_max_plot = max(fv_ann.max(), tan_s) * 1.10
    cal_x = np.linspace(0, x_max_plot, 50)
    cal_y = rf_ann + tan_sharpe * cal_x
    ax.plot(cal_x, cal_y, ':', color=CAL_COLOR, lw=1.6,
            label=f'CAL  (Sharpe = {tan_sharpe:.3f})', zorder=3)

    # ---- Individual asset points (hollow neutral circles, labels offset) ----
    for a in ASSETS:
        s = ret_window[a].std()  * np.sqrt(TRADING_DAYS)
        r = ret_window[a].mean() * TRADING_DAYS
        ax.scatter(s, r, s=70, marker='o', facecolor='white',
                   edgecolors='#333333', lw=1.2, zorder=5)
        ax.annotate(a, (s, r), xytext=ASSET_LABEL_OFFSETS[a], textcoords='offset points',
                    fontsize=9, fontweight='bold', color='#222222')

    # ---- Risk-free anchor (on the y-axis at rf_ann) ----
    ax.scatter(0, rf_ann, marker='s', s=70, c='black', zorder=6)
    ax.annotate(f'$R_f$ = {rf_ann:.2%}', (0, rf_ann),
                xytext=(8, -14), textcoords='offset points', fontsize=9, color='#222222')

    # ---- GMV ----
    gmv_color = PORTFOLIO_COLORS['GMV']
    ax.scatter(gmv_s, gmv_r, marker='*', s=340, c=[gmv_color],
               edgecolors='black', lw=1.0, zorder=7, label=r'GMV  ($σ_{min}$)')
    ax.annotate('GMV', (gmv_s, gmv_r),
                xytext=PORTFOLIO_LABEL_OFFSETS['GMV'], textcoords='offset points',
                fontsize=10, fontweight='bold', color=gmv_color)

    # ---- Tangency ----
    tan_color = PORTFOLIO_COLORS['Tangency']
    ax.scatter(tan_s, tan_r, marker='X', s=280, c=[tan_color],
               edgecolors='black', lw=1.0, zorder=7, label='Tangency')
    ax.annotate('Tangency', (tan_s, tan_r),
                xytext=PORTFOLIO_LABEL_OFFSETS['Tangency'], textcoords='offset points',
                fontsize=10, fontweight='bold', color=tan_color)

    # ---- Optional: training-derived portfolios as HOLLOW markers (holding only) ----
    if extra_committed:
        for lbl, w, marker in extra_committed:
            base_name = lbl.split(' (')[0]
            color = PORTFOLIO_COLORS.get(base_name, '#555555')
            r, s, _ = port_stats_window(w, ret_window)
            ax.scatter(s, r, marker=marker, s=260, facecolor='white',
                       edgecolors=color, lw=2.0, zorder=6, label=lbl)
            # No annotation — legend identifies these; avoids further overlap.

    # ---- Student portfolios ----
    for name, w, marker in STUDENT_PORTFOLIOS:
        color = PORTFOLIO_COLORS[name]
        r, s, _ = port_stats_window(w, ret_window)
        ax.scatter(s, r, marker=marker, s=180, c=[color],
                   edgecolors='black', lw=0.9, zorder=6, label=name)
        ax.annotate(name, (s, r),
                    xytext=PORTFOLIO_LABEL_OFFSETS[name], textcoords='offset points',
                    fontsize=9, fontweight='bold', color=color)

    # ---- Axes & limits (origin anchored at bottom-left) ----
    ax.set_xlabel('Annualized Standard Deviation  (σ)', fontsize=11)
    ax.set_ylabel('Annualized Expected Return  E[r]',    fontsize=11)
    set_titled(ax, 'Long-Only Efficient Frontier', title_subtitle, pad=22)

    y_min_data = min(0.0, fr_ann.min(), min(ret_window[a].mean() * TRADING_DAYS for a in ASSETS))
    y_max_data = max(fr_ann.max(), tan_r, max(ret_window[a].mean() * TRADING_DAYS for a in ASSETS))

    ax.set_xlim(0, x_max_plot)
    ax.set_ylim(y_min_data * 1.10 if y_min_data < 0 else 0, y_max_data * 1.10)

    # ---- Legend: compact two-column at bottom-right ----
    ax.legend(loc='lower right', fontsize=9, frameon=True,
              edgecolor='#666666', facecolor='white', framealpha=0.96,
              ncol=2, handletextpad=0.6, columnspacing=1.2, borderpad=0.6)

    plt.tight_layout()
    savefig(fig, savename)
    plt.show()

    return fv, fr, gmv_r, gmv_s


In [ ]:
# --- Training-window frontier (Fig 1a) ---
front_vol_tr, front_ret_tr, gmv_ret_tr, gmv_vol_tr = plot_frontier(
    returns_train, mu_train_daily, cov_train, rf_train_ann,
    w_gmv, w_tan,
    title_subtitle=f'Training Window: {window_label(returns_train)}',
    savename='fig1a_frontier_training.png',
)


In [ ]:
# --- Holding-window frontier (Fig 1b) ---
# Plot the holding-window frontier (computed on holding-window data) and mark BOTH:
#   - Window-optimal points (filled): what hindsight says the optimizer should have done
#   - Training-derived points (hollow): where w_gmv and w_tan actually landed OOS
front_vol_hd, front_ret_hd, gmv_ret_hd, gmv_vol_hd = plot_frontier(
    returns_hold, mu_hold_daily, cov_hold, rf_hold_ann,
    w_gmv_hold, w_tan_hold,
    title_subtitle=f'Holding Window: {window_label(returns_hold)}',
    savename='fig1b_frontier_holding.png',
    extra_committed=[
        ('GMV (training-derived)',      w_gmv, '*'),
        ('Tangency (training-derived)', w_tan, 'X'),
    ],
)


## 9. Portfolio Statistics Tables

Same statistics as the asset summary, but for each of GMV, tangency, and the four student portfolios. The **weights are fixed** (training-derived for GMV and tangency; constructional for the four student portfolios); only the return window changes.

In [ ]:
def portfolio_stat_row(w, label, ret_window, X_window, rf_ann_window):
    """Full stat row for weight vector w over ret_window."""
    w = np.asarray(w)
    daily = ret_window[ASSETS] @ w

    er_ann = daily.mean() * TRADING_DAYS
    sigma_ann  = daily.std()  * np.sqrt(TRADING_DAYS)
    hpr    = (1 + daily).prod() - 1

    y = daily - ret_window['DAILY_RF']
    res = sm.OLS(y, X_window).fit()
    alpha_ann = res.params['const'] * TRADING_DAYS
    beta_p    = res.params['SPY']

    sharpe = (er_ann - rf_ann_window) / sigma_ann

    return pd.Series({
        'E[r]':   er_ann,
        'σ':      sigma_ann,
        'HPR':    hpr,
        'β':      beta_p,
        'α':      alpha_ann,
        'Sharpe': sharpe,
    }, name=label)

def build_port_table(ret_window, X_window, rf_ann_window):
    rows = [
        portfolio_stat_row(w_gmv,      'GMV',          ret_window, X_window, rf_ann_window),
        portfolio_stat_row(w_tan,      'Tangency',     ret_window, X_window, rf_ann_window),
        portfolio_stat_row(w_bottomup, 'Bottom-Up',    ret_window, X_window, rf_ann_window),
        portfolio_stat_row(w_equal,    'Equal-Weight', ret_window, X_window, rf_ann_window),
        portfolio_stat_row(w_price,    'Price-Weight', ret_window, X_window, rf_ann_window),
        portfolio_stat_row(w_value,    'Value-Weight', ret_window, X_window, rf_ann_window),
    ]
    return pd.concat(rows, axis=1).T

def style_port_table(port_table, subtitle):
    return (port_table.style
        .format({
            'E[r]':   '{:.2%}',
            'σ':      '{:.2%}',
            'HPR':    '{:.2%}',
            'β':      '{:.3f}',
            'α':      '{:.2%}',
            'Sharpe': '{:.3f}',
        })
        .background_gradient(cmap='Blues',  subset=['E[r]', 'HPR'])
        .background_gradient(cmap='Greens', subset=['Sharpe'])
        .set_caption(f'<div style="font-weight:bold;font-size:1.05em;">Portfolio Performance Statistics</div>'
                     f'<div style="font-style:italic;font-size:0.85em;color:#666;">{subtitle}</div>'))


In [ ]:
# --- Training-window portfolio stats (Tab 2a) ---
port_table_train = build_port_table(returns_train, X_train, rf_train_ann)
styled_port_train = style_port_table(port_table_train,
                                     f'Training Window: {window_label(returns_train)}')
with open(os.path.join(OUTDIR, 'tab2a_portfolio_stats_training.html'), 'w') as f:
    f.write(styled_port_train.to_html())
print(f"  saved -> {os.path.join(OUTDIR, 'tab2a_portfolio_stats_training.html')}")
styled_port_train


In [ ]:
# --- Holding-window portfolio stats (Tab 2b) ---
port_table_hold = build_port_table(returns_hold, X_hold, rf_hold_ann)
styled_port_hold = style_port_table(port_table_hold,
                                    f'Holding Window: {window_label(returns_hold)}')
with open(os.path.join(OUTDIR, 'tab2b_portfolio_stats_holding.html'), 'w') as f:
    f.write(styled_port_hold.to_html())
print(f"  saved -> {os.path.join(OUTDIR, 'tab2b_portfolio_stats_holding.html')}")
styled_port_hold


## 10. Figure 2 — Cumulative Wealth (training window)

Growth of $1 in each asset (and SPY) over the training window, compounded daily, log scale. The data the optimizer "saw."

(This is intentionally training-only — its holding-window counterpart is Figure 3, which shows portfolio performance rather than individual assets.)

In [ ]:
asset_cw_train = (1 + returns_train[ASSETS]).cumprod()
spy_cw_train   = (1 + returns_train['SPY']).cumprod()

ASSET_STYLE = {
    'PLTR': dict(color='#3B0F70', linestyle='-',  lw=1.8),
    'NVDA': dict(color='#8C2981', linestyle='-',  lw=1.8),
    'INTC': dict(color='#DE4968', linestyle='--', lw=1.6),
    'GLD':  dict(color='#FE9F6D', linestyle='-.', lw=1.6),
    'EEM':  dict(color='#8C7A00', linestyle=':',  lw=2.0),
}

fig, ax = plt.subplots(figsize=(11, 6))

for a in ASSETS:
    ax.plot(asset_cw_train.index, asset_cw_train[a], label=a, **ASSET_STYLE[a])
ax.plot(spy_cw_train.index, spy_cw_train, label='SPY (market)',
        color='#0066CC', lw=2.2, linestyle='--', alpha=0.85)

ax.axhline(1, color='#888888', lw=0.6)
ax.set_yscale('log')
ax.yaxis.set_major_locator(mticker.FixedLocator([0.5, 1, 2, 5, 10, 20, 50]))
ax.yaxis.set_minor_locator(mticker.NullLocator())
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:g}'))

ax.set_ylabel('Growth of $1 (log scale)', fontsize=11)
set_titled(ax, 'Cumulative Wealth', f'Training Window: {window_label(returns_train)}', pad=20)
ax.legend(loc='upper left', fontsize=9, frameon=True,
          edgecolor='#888888', facecolor='white', framealpha=0.95)

plt.tight_layout()
savefig(fig, 'fig2_cumwealth_training.png')
plt.show()


## 11. Figure 3 — Cumulative Wealth (holding window, out-of-sample)

Growth of $1 invested at the start of the holding window, with training-window weights applied without re-optimization. **Linear scale.** The exam's "and here's what actually happened" centerpiece.

In [ ]:
def hold_cumwealth(weights):
    daily = returns_hold[ASSETS] @ np.asarray(weights)
    return (1 + daily).cumprod()

hold_curves = pd.DataFrame({
    'GMV':          hold_cumwealth(w_gmv),
    'Tangency':     hold_cumwealth(w_tan),
    'Equal-Weight': hold_cumwealth(w_equal),
    'Value-Weight': hold_cumwealth(w_value),
})
spy_cw_hold = (1 + returns_hold['SPY']).cumprod()

HOLD_STYLE = {
    'GMV':          dict(color='#2E7D5B', linestyle='-',  lw=2.0, marker='o', markevery=15, markersize=5),
    'Tangency':     dict(color='#A6324F', linestyle='-',  lw=2.0, marker='X', markevery=15, markersize=6),
    'Equal-Weight': dict(color='#0066CC', linestyle='--', lw=1.8, marker='D', markevery=15, markersize=5),
    'Value-Weight': dict(color='#006633', linestyle='-.', lw=1.8, marker='v', markevery=15, markersize=6),
}

fig, ax = plt.subplots(figsize=(11, 6))

for col in hold_curves.columns:
    ax.plot(hold_curves.index, hold_curves[col], label=col, **HOLD_STYLE[col])
ax.plot(spy_cw_hold.index, spy_cw_hold, label='SPY (benchmark)',
        color='#1F2A6E', lw=2.2, linestyle=':', alpha=0.9)

ax.axhline(1, color='#888888', lw=0.6)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:.2f}'))

ax.set_ylabel('Growth of $1 (linear scale)', fontsize=11)
set_titled(ax, 'Out-of-Sample Cumulative Wealth',
           f'Holding Window: {window_label(returns_hold)}  ·  training-window weights, no re-optimization',
           pad=20)
ax.legend(loc='upper left', fontsize=9, frameon=True,
          edgecolor='#888888', facecolor='white', framealpha=0.95)

plt.tight_layout()
savefig(fig, 'fig3_cumwealth_holding.png')
plt.show()

print('\nFinal wealth at end of holding window:')
print((pd.concat([hold_curves.iloc[-1], pd.Series({'SPY': spy_cw_hold.iloc[-1]})])
       .map(lambda v: f'${v:.4f}')).to_string())


## 12. Figure 4 — Long-Only Frontier Composition

How the optimal weights shift as the target return slides up the efficient frontier. Stacked area, hatched for grayscale legibility. Produced for both windows so students can compare the composition the optimizer would have wanted on training data vs. on holding data.

In [ ]:
ASSET_COLORS_COMP = {
    'PLTR': '#3B0F70',
    'NVDA': '#8C2981',
    'INTC': '#DE4968',
    'GLD':  '#FE9F6D',
    'EEM':  '#FCFDBF',
}
HATCHES = ['', '///', '...', 'xxx', '\\\\']

def plot_composition(mu_daily, cov, w_gmv_win, w_tan_win,
                     title_subtitle, savename):
    eps = 1e-6
    tgt = np.linspace(mu_daily.min() + eps, mu_daily.max() - eps, 80)
    fv, fr, fw = frontier_cvx(mu_daily, cov, tgt, lb=0.0, ub=1.0)

    gmv_ret_daily = w_gmv_win @ mu_daily.values

    eff_mask = fr >= gmv_ret_daily
    eff_rets = fr[eff_mask] * TRADING_DAYS
    eff_ws   = fw[eff_mask]

    sort_idx = np.argsort(eff_rets)
    eff_rets = eff_rets[sort_idx]
    eff_ws   = eff_ws[sort_idx]

    fig, ax = plt.subplots(figsize=(11, 6))

    colors = [ASSET_COLORS_COMP[a] for a in ASSETS]
    polys = ax.stackplot(eff_rets, eff_ws.T, labels=ASSETS, colors=colors,
                         alpha=0.88, edgecolor='white', lw=0.5)
    for poly, hatch in zip(polys, HATCHES):
        poly.set_hatch(hatch)

    gmv_x = gmv_ret_daily * TRADING_DAYS
    tan_x = (w_tan_win @ mu_daily.values) * TRADING_DAYS
    for x, lbl, color in [(gmv_x, 'GMV', '#2E7D5B'), (tan_x, 'Tangency', '#A6324F')]:
        ax.axvline(x, color=color, lw=1.6, linestyle='--', alpha=0.85, zorder=4)
        ax.text(x, 1.02, lbl, color=color, fontsize=10, fontweight='bold',
                ha='center', transform=ax.get_xaxis_transform())

    ax.set_xlim(eff_rets.min(), eff_rets.max())
    ax.set_ylim(0, 1)
    ax.set_xlabel('Target Annualized Return  E[r]', fontsize=11)
    ax.set_ylabel('Portfolio Weight', fontsize=11)

    set_titled(ax, 'Long-Only Frontier Composition', title_subtitle, pad=28)

    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=10, frameon=False)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))

    plt.tight_layout()
    savefig(fig, savename)
    plt.show()

# --- Training ---
plot_composition(mu_train_daily, cov_train, w_gmv, w_tan,
                 title_subtitle=f'Training Window: {window_label(returns_train)}',
                 savename='fig4a_composition_training.png')


In [ ]:
# --- Holding ---
plot_composition(mu_hold_daily, cov_hold, w_gmv_hold, w_tan_hold,
                 title_subtitle=f'Holding Window: {window_label(returns_hold)}',
                 savename='fig4b_composition_holding.png')


## 13. Figure 5 — Security Characteristic Lines + Security Market Line

Two panels in one figure:

- **Left (SCL panel)** — every asset's regression of excess return on SPY excess return, all on one set of axes. Slope = β, intercept = α (daily). Compare slopes directly across assets to see relative market sensitivity.
- **Right (SML panel)** — CAPM prediction line R_f + β·(E[r_m] − R_f) with each asset, the tangency portfolio, and SPY at (β, mean return). Vertical dashes from each point to the SML visualize realized α.

Produced for both training and holding windows.

In [ ]:
ASSET_COLORS_SML = {
    'PLTR': '#3B0F70',
    'NVDA': '#8C2981',
    'INTC': '#DE4968',
    'GLD':  '#B86200',
    'EEM':  '#8C7A00',
}
ASSET_MARKERS = {'PLTR': 'o', 'NVDA': 's', 'INTC': '^', 'GLD': 'D', 'EEM': 'v'}
ASSET_LINESTYLES = {'PLTR': '-', 'NVDA': '--', 'INTC': '-.', 'GLD': ':', 'EEM': (0, (3, 1, 1, 1))}

def plot_scl_sml(ret_window, X_window, rf_ann_window, w_tan_win,
                 betas_list, alphas_ann_list, title_subtitle, savename):
    """
    betas_list        : list of β estimates per asset (already fit on this window)
    alphas_ann_list   : annualized α per asset, same window
    """
    market_excess = (ret_window['SPY'] - ret_window['DAILY_RF']).values
    market_ann = ret_window['SPY'].mean() * TRADING_DAYS
    mkt_premium = market_ann - rf_ann_window

    fig, (ax_scl, ax_sml) = plt.subplots(1, 2, figsize=(16, 6.5))

    # ============================================================
    # LEFT PANEL: SCLs
    # ============================================================
    x_lo, x_hi = market_excess.min(), market_excess.max()
    xx = np.linspace(x_lo, x_hi, 100)

    # Light scatter cloud for each asset (alpha low so lines stand out)
    for a, beta_a in zip(ASSETS, betas_list):
        y_excess = (ret_window[a] - ret_window['DAILY_RF']).values
        ax_scl.scatter(market_excess, y_excess, s=6, alpha=0.18,
                       color=ASSET_COLORS_SML[a], edgecolors='none')

    # Fitted SCL line per asset on top of the cloud
    for a, beta_a, alpha_ann in zip(ASSETS, betas_list, alphas_ann_list):
        alpha_daily = alpha_ann / TRADING_DAYS
        yy = alpha_daily + beta_a * xx
        ax_scl.plot(xx, yy, ASSET_LINESTYLES[a], color=ASSET_COLORS_SML[a],
                    lw=2.2, label=f'{a}: β={beta_a:.2f},  α={alpha_ann:.2%}')

    # Origin guides
    ax_scl.axhline(0, color='#444444', lw=1.0, linestyle='--', alpha=0.7)
    ax_scl.axvline(0, color='#444444', lw=1.0, linestyle='--', alpha=0.7)

    ax_scl.set_xlabel('Market Excess Return  (SPY − $R_f$, daily)', fontsize=11)
    ax_scl.set_ylabel('Asset Excess Return  (daily)', fontsize=11)
    set_titled(ax_scl, 'Security Characteristic Lines', None, pad=14)
    ax_scl.legend(loc='upper left', fontsize=8.5, frameon=True,
                  edgecolor='#888888', facecolor='white', framealpha=0.95)

    # ============================================================
    # RIGHT PANEL: SML
    # ============================================================
    asset_er = {a: ret_window[a].mean() * TRADING_DAYS for a in ASSETS}
    asset_beta = dict(zip(ASSETS, betas_list))

    # Tangency portfolio β + return
    tan_daily = ret_window[ASSETS] @ w_tan_win
    res_tan = sm.OLS(tan_daily - ret_window['DAILY_RF'], X_window).fit()
    tan_beta = res_tan.params['SPY']
    tan_er = tan_daily.mean() * TRADING_DAYS

    beta_max = max(max(asset_beta.values()), tan_beta, 1.0) * 1.15
    beta_grid = np.linspace(0, beta_max, 100)
    sml_y = rf_ann_window + beta_grid * mkt_premium

    y_upper = max(max(asset_er.values()), tan_er, market_ann) * 1.20
    y_lower = min(0, min(asset_er.values()), tan_er) * 1.10

    ax_sml.plot(beta_grid, sml_y, '-', color='#333333', lw=2.0,
                label=f'SML:  $R_f + β(E[r_m] - R_f)$  (slope = {mkt_premium:.2%})')

    ax_sml.fill_between(beta_grid, sml_y, y_upper, color='#88CC88', alpha=0.10, zorder=0)
    ax_sml.fill_between(beta_grid, sml_y, y_lower, color='#CC8888', alpha=0.10, zorder=0)

    for a in ASSETS:
        beta_a = asset_beta[a]
        r = asset_er[a]
        pred = rf_ann_window + beta_a * mkt_premium
        ax_sml.plot([beta_a, beta_a], [r, pred], '--', color='#888888', lw=1.0, alpha=0.7, zorder=2)
        ax_sml.scatter(beta_a, r, s=180, marker=ASSET_MARKERS[a], color=ASSET_COLORS_SML[a],
                       edgecolors='black', lw=1.2, zorder=5, label=a)
        ax_sml.annotate(a, (beta_a, r), xytext=(10, 6), textcoords='offset points',
                        fontsize=10, fontweight='bold')

    # Tangency
    pred_tan = rf_ann_window + tan_beta * mkt_premium
    ax_sml.plot([tan_beta, tan_beta], [tan_er, pred_tan], '--', color='#888888',
                lw=1.0, alpha=0.7, zorder=2)
    ax_sml.scatter(tan_beta, tan_er, marker='X', s=260, color='#A6324F',
                   edgecolors='black', lw=1.2, zorder=6, label='Tangency')
    ax_sml.annotate('Tangency', (tan_beta, tan_er), xytext=(10, -16),
                    textcoords='offset points', fontsize=10, fontweight='bold', color='#A6324F')

    # SPY (market)
    ax_sml.scatter(1.0, market_ann, marker='*', s=260, color='#0066CC',
                   edgecolors='black', lw=1.2, zorder=5, label='SPY (market)')
    ax_sml.annotate('SPY', (1.0, market_ann), xytext=(10, -16),
                    textcoords='offset points', fontsize=10, fontweight='bold', color='#0066CC')

    # Risk-free
    ax_sml.scatter(0, rf_ann_window, marker='s', s=80, color='black', zorder=5)
    ax_sml.annotate(f'$R_f$ = {rf_ann_window:.2%}', (0, rf_ann_window),
                    xytext=(6, -14), textcoords='offset points', fontsize=10)

    ax_sml.set_xlabel('β', fontsize=11)
    ax_sml.set_ylabel('Annualized Mean Return', fontsize=11)
    set_titled(ax_sml, 'Security Market Line',
               'Dashes show realized α (gap between mean return and SML)', pad=14)

    ax_sml.set_xlim(left=0, right=beta_max)
    ax_sml.set_ylim(bottom=y_lower, top=y_upper)
    ax_sml.legend(loc='upper left', fontsize=9, frameon=True,
                  edgecolor='#888888', facecolor='white', framealpha=0.95)

    # Figure-level subtitle (the window label) above both panels
    fig.suptitle(title_subtitle, x=0.02, ha='left', y=1.00,
                 fontsize=10, style='italic', color='#666666')

    plt.tight_layout()
    savefig(fig, savename)
    plt.show()


In [ ]:
# --- Training ---
plot_scl_sml(returns_train, X_train, rf_train_ann, w_tan,
             betas_list=betas_train, alphas_ann_list=alphas_train_ann,
             title_subtitle=f'Training Window: {window_label(returns_train)}',
             savename='fig5a_scl_sml_training.png')


In [ ]:
# --- Holding ---
plot_scl_sml(returns_hold, X_hold, rf_hold_ann, w_tan_hold,
             betas_list=betas_hold, alphas_ann_list=alphas_hold_ann,
             title_subtitle=f'Holding Window: {window_label(returns_hold)}',
             savename='fig5b_scl_sml_holding.png')


## Done

All figures and tables are in `exam_figures/`. Data cache lives in `data_cache/` — delete to force a refresh.

```
data_cache/
  prices_2020-10-01_2026-04-30.parquet
  fred_rf_2020-10-01_2026-04-30.parquet

exam_figures/
  fig1a_frontier_training.png        fig1b_frontier_holding.png
  fig2_cumwealth_training.png        fig3_cumwealth_holding.png
  fig4a_composition_training.png     fig4b_composition_holding.png
  fig5a_scl_sml_training.png         fig5b_scl_sml_holding.png

  tab1a_asset_stats_training.html    tab1b_asset_stats_holding.html
  tab2a_portfolio_stats_training.html tab2b_portfolio_stats_holding.html
  tab3a_cov_training.html             tab3b_cov_holding.html
```
